# Group 3 Risk Model


## Setup


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
STAGGER = 'stagger_3'
T_THRESHOLD = 2.0
N_PCA = 5
CURRENCIES = ['MXN/USD', 'SGD/USD', 'CAD/USD', 'DKK/USD', 'EUR/USD', 'AUD/USD']
SECTOR_TO_ETF = {
    'Communication Services': 'XLC', 'Consumer Discretionary': 'XLY',
    'Consumer Staples': 'XLP', 'Energy': 'XLE', 'Financials': 'XLF',
    'Health Care': 'XLV', 'Industrials': 'XLI', 'Materials': 'XLB',
    'Real Estate': 'XLRE', 'Technology': 'XLK', 'Utilities': 'XLU',
}
SECTORS = [s + '_residual' for s in sorted(SECTOR_TO_ETF.values())]
FACTOR_NAMES = CURRENCIES + ['SPY'] + SECTORS

## 1. Load data


In [3]:
companies = pd.read_csv(DATA / 'companies.csv').set_index('ticker')
returns = pd.read_csv(DATA / f'{STAGGER}_returns.csv', index_col='date', parse_dates=True)
sector_returns = pd.read_csv(DATA / f'{STAGGER}_sector_factors.csv', index_col='date', parse_dates=True)
stocks = companies.index.tolist()
all_factors = returns[CURRENCIES + ['SPY']].join(sector_returns[SECTORS], how='outer')
combined = returns[stocks].join(all_factors, how='outer').sort_index()
missing_by_series = combined.isna().sum().rename('missing_dates')
complete = combined.dropna()
Y = complete[stocks]
F = complete[FACTOR_NAMES]
print(f'Using {len(Y)} of {len(combined)} dates; excluded {len(combined) - len(Y)} incomplete dates.')
print(f'Sample: {Y.index.min().date()} through {Y.index.max().date()}')
display(missing_by_series[missing_by_series > 0].to_frame())
display(Y.head())

Using 70 of 80 dates; excluded 10 incomplete dates.
Sample: 2021-05-10 through 2026-08-24


,missing_dates
DOCN,10
UPST,6


,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,VLO,...,LLY,WELL,NOK,MSFT,GOOGL,COST,HD,KO,JNJ,BRK-B
date,,,,,,,,,,,,,,,,,,,,,
2021-05-10,-0.052046,0.044154,-0.007505,0.008636,-0.047057,-0.077541,-0.033385,0.016027,0.115540,0.078888,...,0.025370,-0.015636,0.099490,-0.030982,-0.001200,0.024057,-0.008550,0.012127,0.030059,0.041328
2021-06-07,0.081721,0.191130,-0.014808,0.010341,0.013311,0.004938,0.023957,0.007990,-0.007825,0.000625,...,0.084356,0.053682,0.065208,0.025908,0.040958,-0.004292,-0.022424,0.016376,-0.015980,-0.008488
2021-07-05,0.028692,0.011847,0.004768,-0.036604,-0.001235,0.098312,0.067955,-0.048074,-0.061466,-0.075467,...,0.030583,0.052362,0.004726,0.048212,0.020900,0.049547,0.023038,-0.014959,0.018432,-0.014370
2021-08-02,-0.026209,0.122179,0.029532,0.019811,0.000332,0.110970,-0.069289,-0.014063,0.030391,-0.045271,...,0.074635,-0.009514,0.071191,0.026522,0.051120,0.043028,0.015528,0.025632,0.012800,0.011859
2021-08-30,0.019342,-0.014462,0.013395,0.045026,0.014315,0.011434,0.025847,0.002758,-0.013286,-0.018175,...,-0.007769,0.016865,-0.006547,0.027468,0.037934,0.033661,0.004362,0.001052,0.011268,-0.008730


## 2. Regression helpers


In [4]:
def ols(y, x):
    # No intercept: residuals retain the unexplained return level.
    if x.shape[1] == 0:
        # With no retained factors, there is no regression or t-statistic.
        return dict(beta=pd.Series(dtype=float, index=x.columns),
                    tstat=pd.Series(dtype=float, index=x.columns),
                    residual=y.copy(), nobs=len(y))
    fit = sm.OLS(y.astype(float), x.astype(float), hasconst=False).fit()
    return dict(beta=fit.params, tstat=fit.tvalues,
                residual=fit.resid, nobs=int(fit.nobs))

def backward(y, x, threshold=T_THRESHOLD):
    active = list(x.columns)
    audit = []
    while active:
        fit = ols(y, x[active])
        weakest = fit['tstat'].abs().idxmin()
        drop = weakest if abs(fit['tstat'][weakest]) <= threshold else None
        audit.extend(dict(iteration=len(x.columns) - len(active) + 1, factor=f, beta=fit['beta'][f],
                          tstat=fit['tstat'][f], removed=(f == drop)) for f in active)
        if drop is None:
            return fit, audit
        active.remove(drop)
    return ols(y, x[[]]), audit

betas18 = pd.DataFrame(0.0, index=stocks, columns=FACTOR_NAMES)
prior_records, selection_records = [], []

def record_fit(ticker, block, fit):
    betas18.loc[ticker, fit['beta'].index] = fit['beta']
    prior_records.extend(dict(ticker=ticker, block=block, factor=f,
                              beta=fit['beta'][f], tstat=fit['tstat'][f],
                              nobs=fit['nobs']) for f in fit['beta'].index)

def record_selection(ticker, block, audit):
    selection_records.extend(dict(ticker=ticker, block=block, **row) for row in audit)

## 3. Home currency regressions


In [5]:
after_home_currency = Y.copy()
home_currency = {}
for ticker in stocks:
    currency = companies.loc[ticker, 'currency']
    home_currency[ticker] = None if currency == 'USD' else f'{currency}/USD'
    if home_currency[ticker] is not None:
        fit = ols(Y[ticker], F[[home_currency[ticker]]])
        record_fit(ticker, 'home_currency', fit)
        after_home_currency[ticker] = fit['residual']
display(pd.DataFrame(prior_records))

,ticker,block,factor,beta,tstat,nobs
0,VIST,home_currency,MXN/USD,0.725965,1.233959,70
1,STX,home_currency,SGD/USD,4.057249,2.852572,70
2,NTR,home_currency,CAD/USD,1.264755,2.505925,70
3,ASND,home_currency,DKK/USD,0.587185,0.972130,70
4,NOK,home_currency,EUR/USD,0.900254,1.152717,70


## 4. Australian dollar regression

In [6]:
after_currencies = after_home_currency.copy()
for ticker in stocks:
    fit, audit = backward(after_home_currency[ticker], F[['AUD/USD']])
    record_fit(ticker, 'aud_currency', fit)
    record_selection(ticker, 'aud_currency', audit)
    after_currencies[ticker] = fit['residual']

display(betas18[CURRENCIES])

,MXN/USD,SGD/USD,CAD/USD,DKK/USD,EUR/USD,AUD/USD
VST,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
VIST,0.725965,0.000000,0.000000,0.000000,0.000000,0.000000
AEP,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ETR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
DIS,0.000000,0.000000,0.000000,0.000000,0.000000,1.222460
CMG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AMZN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
FIX,0.000000,0.000000,0.000000,0.000000,0.000000,1.343907
STX,0.000000,4.057249,0.000000,0.000000,0.000000,1.678837
VLO,0.000000,0.000000,0.000000,0.000000,0.000000,1.055426


## 5. Market regression


In [7]:
after_market = Y.copy()
for ticker in stocks:
    fit = ols(after_currencies[ticker], F[['SPY']])
    record_fit(ticker, 'market', fit)
    after_market[ticker] = fit['residual']
display(betas18[['SPY']])

,SPY
VST,1.521770
VIST,0.567034
AEP,-0.006426
ETR,0.299355
DIS,1.080766
CMG,1.205334
AMZN,1.411830
FIX,1.222771
STX,1.787684
VLO,0.354074


## 6. Own sector regression


In [8]:
after_own_sector = Y.copy()
own_sector = {}
for ticker in stocks:
    own_sector[ticker] = SECTOR_TO_ETF[companies.loc[ticker, 'sector']] + '_residual'
    fit = ols(after_market[ticker], F[[own_sector[ticker]]])
    record_fit(ticker, 'own_sector', fit)
    after_own_sector[ticker] = fit['residual']
display(pd.DataFrame(prior_records).query("block == 'own_sector'"))

,ticker,block,factor,beta,tstat,nobs
65,VST,own_sector,XLU_residual,1.279815,4.122523,70
66,VIST,own_sector,XLE_residual,1.044872,5.746156,70
67,AEP,own_sector,XLU_residual,1.104229,11.929446,70
68,ETR,own_sector,XLU_residual,1.105315,10.824582,70
69,DIS,own_sector,XLC_residual,0.758844,2.555393,70
70,CMG,own_sector,XLY_residual,1.149709,3.115777,70
71,AMZN,own_sector,XLY_residual,1.046795,4.068415,70
72,FIX,own_sector,XLI_residual,1.409967,3.214802,70
73,STX,own_sector,XLK_residual,1.666599,3.406213,70
74,VLO,own_sector,XLE_residual,1.031399,9.018571,70


## 7. Remaining sectors regression


In [9]:
residuals18 = Y.copy()
for ticker in stocks:
    candidates = [f for f in SECTORS if f != own_sector[ticker]]
    fit, audit = backward(after_own_sector[ticker], F[candidates])
    record_fit(ticker, 'remaining_sectors', fit)
    record_selection(ticker, 'remaining_sectors', audit)
    residuals18[ticker] = fit['residual']
selection_audit = pd.DataFrame(selection_records)
retained_fits = pd.DataFrame(prior_records)
display(betas18)
display(selection_audit[selection_audit['removed']].head(20))

,MXN/USD,SGD/USD,CAD/USD,DKK/USD,EUR/USD,AUD/USD,SPY,XLB_residual,XLC_residual,XLE_residual,XLF_residual,XLI_residual,XLK_residual,XLP_residual,XLRE_residual,XLU_residual,XLV_residual,XLY_residual
VST,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.521770,0.000000,0.000000,0.000000,0.000000,1.327614,0.000000,0.000000,-1.288875,1.279815,-0.999218,0.000000
VIST,0.725965,0.000000,0.000000,0.000000,0.000000,0.000000,0.567034,-0.825704,0.000000,1.044872,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AEP,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.006426,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.104229,0.269127,0.000000
ETR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.299355,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.105315,0.000000,0.000000
DIS,0.000000,0.000000,0.000000,0.000000,0.000000,1.222460,1.080766,0.000000,0.758844,0.000000,0.706258,0.000000,0.000000,0.000000,-0.557115,0.000000,0.000000,0.000000
CMG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.205334,0.769119,0.000000,-0.371397,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.149709
AMZN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.411830,0.000000,0.000000,0.000000,0.000000,-0.544177,0.000000,-0.546785,0.000000,0.000000,0.000000,1.046795
FIX,0.000000,0.000000,0.000000,0.000000,0.000000,1.343907,1.222771,-0.716335,-1.389553,0.000000,0.000000,1.409967,0.000000,0.000000,0.000000,0.971613,-1.364456,0.000000
STX,0.000000,4.057249,0.000000,0.000000,0.000000,1.678837,1.787684,0.000000,0.000000,0.000000,0.000000,1.218639,1.666599,-1.107192,0.000000,0.000000,0.000000,0.000000
VLO,0.000000,0.000000,0.000000,0.000000,0.000000,1.055426,0.354074,0.000000,0.000000,1.031399,0.000000,0.000000,0.000000,-0.691735,0.000000,0.000000,0.000000,0.000000


,ticker,block,iteration,factor,beta,tstat,removed
0,VST,aud_currency,1,AUD/USD,1.044140,1.644770,True
1,VIST,aud_currency,1,AUD/USD,0.437168,0.674020,True
2,AEP,aud_currency,1,AUD/USD,0.203319,0.767211,True
3,ETR,aud_currency,1,AUD/USD,0.253392,0.903423,True
5,CMG,aud_currency,1,AUD/USD,0.928907,1.787749,True
6,AMZN,aud_currency,1,AUD/USD,0.812007,1.863935,True
10,CF,aud_currency,1,AUD/USD,0.317497,0.612957,True
11,NTR,aud_currency,1,AUD/USD,0.287594,0.799524,True
12,LNN,aud_currency,1,AUD/USD,0.586858,1.610058,True
14,ASND,aud_currency,1,AUD/USD,0.453547,0.897147,True


## 8. Residual covariance matrix


In [10]:
residual_covariance = residuals18.cov()
display(residual_covariance)

,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,VLO,...,LLY,WELL,NOK,MSFT,GOOGL,COST,HD,KO,JNJ,BRK-B
VST,0.007992,0.001309,-0.000979,-0.000704,-0.000044,0.000915,-0.000278,0.000667,-0.000154,0.000386,...,0.000574,-0.000155,-0.001996,0.001107,-9.814783e-04,0.000854,0.001074,-0.000195,-0.000765,-0.000600
VIST,0.001309,0.010303,-0.000333,-0.000492,0.000798,0.001032,0.001040,0.000060,-0.000519,-0.001429,...,0.002545,-0.000571,0.003630,0.000294,-8.270197e-04,0.000646,-0.001555,0.000167,-0.000788,0.000036
AEP,-0.000979,-0.000333,0.000854,0.000377,0.000185,-0.000965,0.000231,0.000021,0.000205,0.000094,...,0.000180,0.000624,0.000342,-0.000597,3.656069e-04,0.000093,-0.000211,0.000108,0.000126,0.000162
ETR,-0.000704,-0.000492,0.000377,0.001144,0.000379,-0.000908,-0.000021,0.000764,0.000874,0.000048,...,-0.000565,0.000370,0.000872,-0.000616,3.895088e-04,-0.000002,-0.000101,0.000062,0.000168,0.000075
DIS,-0.000044,0.000798,0.000185,0.000379,0.003914,0.000777,0.000358,0.001054,0.000242,0.000483,...,-0.000326,0.000063,0.000100,0.000105,-6.651561e-04,0.000205,-0.000405,-0.000070,-0.000136,-0.000235
CMG,0.000915,0.001032,-0.000965,-0.000908,0.000777,0.007780,0.001527,-0.001652,-0.002326,0.000634,...,-0.000060,-0.000950,-0.001436,0.001321,-8.850589e-04,0.000400,-0.000055,-0.000142,-0.001171,-0.000327
AMZN,-0.000278,0.001040,0.000231,-0.000021,0.000358,0.001527,0.003412,0.000188,-0.000286,-0.000212,...,0.000136,0.000147,-0.000846,0.000536,4.056686e-04,0.000435,-0.000835,0.000313,-0.000342,0.000218
FIX,0.000667,0.000060,0.000021,0.000764,0.001054,-0.001652,0.000188,0.007342,0.003528,0.000706,...,-0.000590,0.000140,0.000768,-0.000378,8.306297e-04,0.000302,0.000027,-0.000153,0.000112,0.000239
STX,-0.000154,-0.000519,0.000205,0.000874,0.000242,-0.002326,-0.000286,0.003528,0.013131,0.001359,...,-0.000124,0.001435,0.001474,-0.002025,2.083667e-03,-0.000491,-0.000282,0.000334,0.000827,-0.000908
VLO,0.000386,-0.001429,0.000094,0.000048,0.000483,0.000634,-0.000212,0.000706,0.001359,0.003859,...,-0.001188,0.000464,-0.000447,-0.000405,3.946318e-04,0.000089,0.000823,0.000451,0.000276,-0.000343


## 9. PCA


In [11]:
eigenvalues, eigenvectors = np.linalg.eigh(residual_covariance.to_numpy())
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
for j in range(eigenvectors.shape[1]):
    anchor = np.argmax(np.abs(eigenvectors[:, j]))
    if eigenvectors[anchor, j] < 0:
        eigenvectors[:, j] *= -1
pca_summary = pd.DataFrame({
    'eigenvalue': eigenvalues,
    'explained_variance_ratio': eigenvalues / eigenvalues.sum(),
    'cumulative_explained_variance': np.cumsum(eigenvalues) / eigenvalues.sum(),
    'retained': np.arange(len(eigenvalues)) < N_PCA,
}, index=pd.Index([f'PC{i+1}' for i in range(43)], name='component'))
all_eigenvectors = pd.DataFrame(eigenvectors, index=stocks, columns=pca_summary.index)
pca_loadings = all_eigenvectors.iloc[:, :N_PCA].copy()
centered_residuals = residuals18 - residuals18.mean()
pca_returns = centered_residuals @ pca_loadings
display(pca_summary)
print(f'Five PCs explain {pca_summary.iloc[N_PCA-1].cumulative_explained_variance:.2%} of residual variance.')

,eigenvalue,explained_variance_ratio,cumulative_explained_variance,retained
component,,,,
PC1,0.077136,0.170525,0.170525,True
PC2,0.063709,0.140841,0.311366,True
PC3,0.050424,0.111472,0.422837,True
PC4,0.038702,0.085558,0.508395,True
PC5,0.030363,0.067123,0.575518,True
PC6,0.025806,0.057049,0.632567,False
PC7,0.019239,0.042531,0.675098,False
PC8,0.017090,0.037780,0.712878,False
PC9,0.015671,0.034643,0.747521,False


Five PCs explain 57.55% of residual variance.


## 10. Exposures and factor covariance matrix

In [12]:
exposures23 = pd.concat([betas18, pca_loadings], axis=1)
factor_returns23 = pd.concat([F, pca_returns], axis=1)
factor_covariance23 = factor_returns23.cov()
display(exposures23)
display(factor_covariance23)

,MXN/USD,SGD/USD,CAD/USD,DKK/USD,EUR/USD,AUD/USD,SPY,XLB_residual,XLC_residual,XLE_residual,...,XLP_residual,XLRE_residual,XLU_residual,XLV_residual,XLY_residual,PC1,PC2,PC3,PC4,PC5
VST,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.521770,0.000000,0.000000,0.000000,...,0.000000,-1.288875,1.279815,-0.999218,0.000000,-0.021402,-0.126483,0.028664,0.008943,0.066430
VIST,0.725965,0.000000,0.000000,0.000000,0.000000,0.000000,0.567034,-0.825704,0.000000,1.044872,...,0.000000,0.000000,0.000000,0.000000,0.000000,-0.011310,0.045744,0.129120,-0.051232,0.330006
AEP,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.006426,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.104229,0.269127,0.000000,-0.006725,0.007704,0.007562,0.015187,-0.039897
ETR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.299355,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.105315,0.000000,0.000000,-0.021110,0.004173,-0.011941,0.037647,-0.055725
DIS,0.000000,0.000000,0.000000,0.000000,0.000000,1.222460,1.080766,0.000000,0.758844,0.000000,...,0.000000,-0.557115,0.000000,0.000000,0.000000,0.012750,0.013570,0.015906,0.070530,0.070789
CMG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.205334,0.769119,0.000000,-0.371397,...,0.000000,0.000000,0.000000,0.000000,1.149709,0.094928,0.049202,-0.050038,-0.110953,0.127442
AMZN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.411830,0.000000,0.000000,0.000000,...,-0.546785,0.000000,0.000000,0.000000,1.046795,0.004247,0.071847,0.038905,-0.085341,0.057742
FIX,0.000000,0.000000,0.000000,0.000000,0.000000,1.343907,1.222771,-0.716335,-1.389553,0.000000,...,0.000000,0.000000,0.971613,-1.364456,0.000000,-0.098316,0.038672,0.012848,0.118992,-0.021192
STX,0.000000,4.057249,0.000000,0.000000,0.000000,1.678837,1.787684,0.000000,0.000000,0.000000,...,-1.107192,0.000000,0.000000,0.000000,0.000000,-0.273270,0.161094,-0.017222,0.217683,-0.076086
VLO,0.000000,0.000000,0.000000,0.000000,0.000000,1.055426,0.354074,0.000000,0.000000,1.031399,...,-0.691735,0.000000,0.000000,0.000000,0.000000,0.001453,0.062521,-0.056865,0.100320,-0.028468


,MXN/USD,SGD/USD,CAD/USD,DKK/USD,EUR/USD,AUD/USD,SPY,XLB_residual,XLC_residual,XLE_residual,...,XLP_residual,XLRE_residual,XLU_residual,XLV_residual,XLY_residual,PC1,PC2,PC3,PC4,PC5
MXN/USD,0.000806,0.000153,1.359454e-04,0.000283,0.000263,0.000359,0.000447,0.000393,0.000080,0.000238,...,0.000168,0.000028,-0.000014,-0.000049,0.000019,-4.730140e-04,6.483983e-04,-1.221925e-03,-1.507371e-03,-2.502687e-04
SGD/USD,0.000153,0.000183,1.349420e-04,0.000235,0.000226,0.000246,0.000170,0.000294,0.000053,0.000017,...,0.000215,0.000180,0.000122,0.000053,-0.000029,4.863692e-04,-7.117902e-04,-3.660618e-04,-6.666947e-04,-2.335375e-04
CAD/USD,0.000136,0.000135,3.395687e-04,0.000155,0.000151,0.000316,0.000235,0.000153,0.000118,0.000371,...,0.000142,0.000070,-0.000026,-0.000127,-0.000026,6.592698e-04,-1.108203e-03,-1.239661e-04,-2.378786e-04,-2.201414e-04
DKK/USD,0.000283,0.000235,1.552915e-04,0.000471,0.000458,0.000310,0.000176,0.000375,0.000036,-0.000138,...,0.000264,0.000123,0.000054,0.000067,-0.000067,-8.784432e-05,-8.979190e-04,-9.233959e-04,-1.724879e-03,-9.289420e-04
EUR/USD,0.000263,0.000226,1.511100e-04,0.000458,0.000448,0.000298,0.000156,0.000360,0.000033,-0.000112,...,0.000257,0.000120,0.000063,0.000071,-0.000082,2.875739e-05,-8.056223e-04,-9.690980e-04,-1.607524e-03,-8.825647e-04
AUD/USD,0.000359,0.000246,3.161128e-04,0.000310,0.000298,0.000665,0.000510,0.000468,0.000091,0.000471,...,0.000202,0.000237,0.000150,-0.000044,-0.000052,9.786983e-04,-7.206835e-04,-2.430485e-04,-1.305896e-03,-1.917365e-04
SPY,0.000447,0.000170,2.349117e-04,0.000176,0.000156,0.000510,0.001730,-0.000017,0.000013,-0.000126,...,-0.000027,-0.000014,-0.000010,-0.000030,0.000010,6.146882e-04,-5.101545e-04,4.700383e-04,-2.654108e-04,-2.925836e-04
XLB_residual,0.000393,0.000294,1.528317e-04,0.000375,0.000360,0.000468,-0.000017,0.001933,-0.000059,0.001494,...,0.000760,0.000754,0.000798,0.000664,-0.000201,-1.577488e-04,-4.486829e-04,-1.329876e-03,1.611560e-04,-3.833216e-04
XLC_residual,0.000080,0.000053,1.179925e-04,0.000036,0.000033,0.000091,0.000013,-0.000059,0.000817,-0.000026,...,-0.000118,-0.000095,-0.000014,-0.000253,0.000264,1.627353e-03,-1.533215e-03,-3.090939e-04,4.941075e-04,-1.071805e-04
XLE_residual,0.000238,0.000017,3.714094e-04,-0.000138,-0.000112,0.000471,-0.000126,0.001494,-0.000026,0.005582,...,0.000209,0.000456,0.000510,0.000133,-0.000640,-3.238141e-03,-6.190725e-04,-9.802164e-04,1.582410e-03,1.730582e-03


In [13]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(pca_summary)

,eigenvalue,explained_variance_ratio,cumulative_explained_variance,retained
component,,,,
PC1,0.077136,0.170525,0.170525,True
PC2,0.063709,0.140841,0.311366,True
PC3,0.050424,0.111472,0.422837,True
PC4,0.038702,0.085558,0.508395,True
PC5,0.030363,0.067123,0.575518,True
PC6,0.025806,0.057049,0.632567,False
PC7,0.019239,0.042531,0.675098,False
PC8,0.017090,0.037780,0.712878,False
PC9,0.015671,0.034643,0.747521,False


In [15]:
stock_covariance = Y.cov()

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(stock_covariance)

,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,VLO,CF,NTR,LNN,WDC,ASND,DOCN,CME,NET,MRNA,AXON,NFLX,CMCSA,MU,NEM,PGR,ORKA,UPST,PG,F,HPE,BA,HII,AMD,LLY,WELL,NOK,MSFT,GOOGL,COST,HD,KO,JNJ,BRK-B
VST,0.018322,0.003718,0.000910,0.002626,0.004621,0.004783,0.002996,0.009825,0.006401,0.002713,0.003608,0.002580,0.002389,0.007857,-0.000886,0.001724,-0.000053,0.006963,-0.002756,0.008126,0.003910,0.001401,0.005792,-0.000894,0.001175,-0.000428,0.011867,0.000843,0.003974,0.005834,0.002465,-0.001879,0.005344,-0.001244,0.001591,-0.001318,0.002395,0.001257,0.003097,0.004351,-0.000551,-0.000934,-0.000191
VIST,0.003718,0.018011,-0.000352,-0.000058,0.002599,-0.000968,0.001252,0.002989,0.002167,0.005604,0.006003,0.003380,-0.001097,0.003916,-0.005355,0.004500,0.002228,0.005400,-0.004085,0.000004,0.004063,0.000446,0.002814,-0.004757,0.000006,0.006101,0.001091,-0.000350,0.005586,0.005974,-0.001209,0.001663,-0.001246,0.003074,-0.000225,0.003920,0.001345,-0.000180,0.000990,-0.002631,0.000223,-0.000209,0.001075
AEP,0.000910,-0.000352,0.003171,0.002527,-0.000485,-0.001111,-0.001626,0.001641,-0.000146,-0.000045,0.000056,0.000575,0.000736,0.000227,0.001173,-0.000838,0.000018,-0.001735,0.000701,-0.001392,-0.000785,0.000467,-0.000957,0.000754,0.000972,0.000905,-0.001781,0.001058,-0.000603,-0.002827,-0.000001,0.000927,-0.000219,0.000274,0.002091,0.000487,-0.001980,-0.000809,0.000437,0.000928,0.001365,0.001757,0.000700
ETR,0.002626,-0.000058,0.002527,0.003418,0.000710,-0.000249,-0.000717,0.003454,0.001578,0.000349,0.000729,0.000549,0.000890,0.001696,0.000750,-0.000222,0.000163,-0.000746,0.000389,-0.000897,0.000261,0.000861,0.000372,0.000853,0.000612,0.000449,0.000505,0.000920,0.000020,-0.000849,0.000858,0.000408,0.000619,-0.001137,0.001719,0.001093,-0.001502,-0.000342,0.000573,0.001507,0.000796,0.001365,0.000595
DIS,0.004621,0.002599,-0.000485,0.000710,0.008612,0.005062,0.004372,0.003649,0.005995,0.002297,0.001720,0.001217,0.001933,0.005106,0.001507,0.006985,0.000888,0.008267,0.007267,0.007342,0.006541,0.002297,0.007329,0.000927,0.001177,0.002206,0.012616,0.000220,0.003324,0.004601,0.004909,-0.001444,0.005134,-0.000156,0.001195,0.002494,0.002827,0.001888,0.001985,0.001911,-0.000395,-0.000338,0.000534
CMG,0.004783,-0.000968,-0.001111,-0.000249,0.005062,0.012963,0.005248,0.000984,0.001710,-0.000646,-0.000021,0.000764,0.002087,0.001689,0.002688,0.001188,-0.000407,0.003805,0.010253,0.008353,0.005037,0.002693,0.004776,0.001333,0.000821,0.005552,0.012622,0.000579,0.003033,0.001617,0.005986,0.000482,0.004626,0.000079,-0.000322,0.000410,0.002735,0.000839,0.002663,0.003541,-0.000024,-0.001014,0.000441
AMZN,0.002996,0.001252,-0.001626,-0.000717,0.004372,0.005248,0.009043,0.003385,0.004945,-0.000116,0.000862,0.000225,-0.000836,0.004914,0.001987,0.008412,0.000600,0.007181,0.000562,0.004070,0.006008,-0.000626,0.004994,-0.000626,-0.001451,0.006687,0.011914,-0.001098,0.002276,0.003108,0.004670,-0.000751,0.007998,-0.000093,0.000152,0.001448,0.004055,0.005149,0.002379,0.000589,-0.000985,-0.002020,-0.000125
FIX,0.009825,0.002989,0.001641,0.003454,0.003649,0.000984,0.003385,0.017479,0.011502,0.002223,0.003898,0.001170,0.000489,0.012151,0.000611,0.010164,0.000537,0.003251,-0.005119,0.001833,0.000604,-0.000088,0.009656,-0.000138,-0.001157,0.005046,0.006090,0.000679,0.004021,0.004616,0.004066,0.001795,0.012824,-0.002914,0.000894,0.002617,0.001509,0.003565,0.002184,0.002921,-0.000416,-0.000407,0.000089
STX,0.006401,0.002167,-0.000146,0.001578,0.005995,0.001710,0.004945,0.011502,0.026120,0.004475,0.001495,0.000602,0.002414,0.023272,0.000632,0.021772,-0.000161,0.009642,0.003198,-0.001271,-0.000892,0.000868,0.022271,0.003319,-0.001220,0.008240,0.008527,0.000549,0.007294,0.013685,0.007421,-0.002316,0.017756,0.000492,0.002485,0.009463,0.002591,0.006722,0.001772,0.002455,0.000597,0.000449,-0.000643
VLO,0.002713,0.005604,-0.000045,0.000349,0.002297,-0.000646,-0.000116,0.002223,0.004475,0.011023,0.006184,0.003886,0.000720,0.005888,-0.000908,0.000902,0.000770,0.002236,0.003703,0.000325,0.000787,0.001